# Adding the AF2 Facebook Distillation Dataset

In this example, we demonstrate how to add a distillation dataset to DataHub using the AlphaFold2 Facebook distillation dataset. This dataset contains predicted protein structures and associated metadata.

## Dataset Overview

The AF2 Facebook distillation dataset includes:
- Predicted protein structures in CIF format -- these were predicted by AF2
- Metadata for each structure (e.g., sequence length, pLDDT scores)
- MSA files
- Templates

## Steps to Add the Dataset

1. **Load Metadata**: 
   We use `pandas` to read the metadata from a `parquet` file, selecting relevant columns.

2. **Prepare dataframe to be compatible with the GenericDFParser**:
   The easiest way to load a dataset is with the `GenericDFParser` (see `datahub/datasets/parsers/default_metadata_row_parsers` for more information)

   We require columns that:
   - Include a `path` column that tells us where to find the CIF file
   - Includes a unique `example_id` column, formatted according to our convention (defined in `datahub/common/generate_example_id`)
   
   Typically, we will also want our dataframe to contain columns that:
   - Designate the `pn_unit_iid` (`chain_id`) of the molecule we want to center our crop around (if not given, we will choose a random atom)
   - Provides default values for `assembly_id`, since we usually won't have multiple assemblies when training on examples that are not from the PDB

   > **NOTE**: We no longer suggest making a custom `MetadataRowParser` for each dataset; instead, please format your dataframe according to our `GenericDFParser` to avoid a proliferation of similar parsers.

3. **Initialize Dataset Objects**:
   - Create a `PandasDataset` with the metadata
   - Wrap it in a `StructuralDatasetWrapper` with our generic parser

## Usage

After setting up the dataset, you can use it like any other DataHub dataset.


## Setup
So let's get started by importing the necessary libraries and setting up our environment.

In [1]:
import pandas as pd
from pathlib import Path

from datahub.datasets.datasets import PandasDataset
# ^-- this is the base dataset class to hold the metadata table

from datahub.datasets.parsers.default_metadata_row_parsers import GenericDFParser
# ^-- this class implements the abstract class MetadataRowParser, which says how to assemble the paths from the metadata in table

from datahub.datasets.datasets import StructuralDatasetWrapper
# ^-- this is a wrapper to jointly hold the metadata (which examples we have), the row parser (how to get the data
#     associated with each example), the CIF parser (how to parse the CIF file once we have the path) and the
#     transforms that should be applied to the data once we have it to featurize it.

from cifutils import parse
# ^-- this is the parse function for structural data. It can parse all types of files - `cif`, `pdb`, etc.

In [2]:
AF2_DISTILLATION_DIR = "/squash/af2_distillation_facebook/"
# ^ a squashed directory with all the data (does not change, so we store in a squash)
AF2_DISTILLATION_PARQUET = (
    "/projects/ml/datahub/dfs/distillation/af2_distillation_facebook/af2_distillation_facebook.parquet"
)
# ^ a parquet file with the metadata (may be updated periodically, so we store in a parquet on DIGS)

In [3]:
# Let's see if we've got the right directory and what to expect there
!ls -l {AF2_DISTILLATION_DIR}
# Let's look at the README file to get an idea of what's in the dataset

total 2064500
-rw-r--r--   1 smathis baker 2114045232 Sep 18 01:48 af2_distillation_facebook.parquet
drwxr-xr-x 258 smathis baker       5635 Sep 16 19:57 cif
drwxr-xr-x 258 smathis baker       5635 Sep 16 19:57 msa
-rw-r--r--   1 smathis baker       1542 Sep 16 22:29 README
drwxr-xr-x 258 smathis baker       5635 Sep 16 19:57 template


In [4]:
# Let's look at the README file to get an idea of what's in the dataset
!cat {AF2_DISTILLATION_DIR}/README

---------------------------------------------------------------------------
AF2 Distillation Dataset (provided courtesy ESM Facebook research)
---------------------------------------------------------------------------

 This directory contains AF2 predictions, MSAs and templates for ~7.6 Mio
 clusters in UniRef50, courtesy Facebook Research.

 Metadata (i.e. which sequences, which cluster identities @ 30% seq.id, 
 whether a sequence has an msa & template, sequence_hash etc.) are stored
 in the `af2_distillation_facebook.parquet` dataframe. You can load this 
 one with pandas.

 The `cif`, `msa` and `template` directories are sharded in two layers
 by the first two characters of the sequence_hash and contain the AF2
 model predictions, .a3m msa files and .atab template files from HHR
 respectively.
 
 The two layers of sharding is an optimization technique for faster filesystem
 performance. (Do not put more than 10k files in any directory).

 Example: 
  - example_id: UniRef50_A0A1S3

Next, let's load the metadata table. We will load everything except the sequence column, which is pretty large 
and we don't need it anyways in the metadata, since we have the sequence hash already provided and we could 
always get the sequence back from the cif file.

In [5]:
df_metadata = pd.read_parquet(
    AF2_DISTILLATION_PARQUET,
    columns=[
        "path",
        "example_id",
        "original_hash_for_shard",
        "n_atoms",
        "n_res",
        "mean_plddt",
        "min_plddt",
        "median_plddt",
        "sequence_hash",
        "has_msa",
        "msa_depth",
        "has_template",
        "cluster_id",
        # NOTE: The `seq` column is very large, so we don't include it here.
    ],
)

Let's look at the first few rows of the metadata table.

In [6]:
df_metadata

,path,example_id,original_hash_for_shard,n_atoms,n_res,mean_plddt,min_plddt,median_plddt,sequence_hash,has_msa,msa_depth,has_template,cluster_id
0,/squash/af2_distillation_facebook/cif/60/84/Un...,{['af2_distillation_facebook']}{UniRef50_A0A0D...,8bca,1402,187,88.808050,44.733899,92.602713,608400c62c4,True,1517,True,789965
1,/squash/af2_distillation_facebook/cif/68/a6/Un...,{['af2_distillation_facebook']}{UniRef50_A0A51...,0b5c,877,113,96.507410,78.776705,97.047538,68a67b8d9bc,True,1076,True,1629745
2,/squash/af2_distillation_facebook/cif/68/25/Un...,{['af2_distillation_facebook']}{UniRef50_UPI00...,5976,1603,201,90.922863,63.803520,93.433699,68254446ed1,True,1357,True,1842918
3,/squash/af2_distillation_facebook/cif/4a/d5/Un...,{['af2_distillation_facebook']}{UniRef50_A0A38...,e7c5,1341,175,92.490567,65.324014,97.594444,4ad59972a4d,True,2382,True,725087
4,/squash/af2_distillation_facebook/cif/2f/0f/Un...,{['af2_distillation_facebook']}{UniRef50_Q12KH...,ba5d,1196,149,85.225776,52.848206,87.366418,2f0ffffd1f0,True,1166,True,2564944
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7597989,/squash/af2_distillation_facebook/cif/b2/3c/Un...,{['af2_distillation_facebook']}{UniRef50_A0A14...,51fe,2205,277,91.454213,46.115079,95.574151,b23ce115ba1,True,2564,True,2223658
7597990,/squash/af2_distillation_facebook/cif/36/6c/Un...,{['af2_distillation_facebook']}{UniRef50_A0A3C...,f0cc,3020,393,85.714320,24.754866,91.117117,366c79c71bd,True,3009,True,2668728
7597991,/squash/af2_distillation_facebook/cif/20/db/Un...,{['af2_distillation_facebook']}{UniRef50_A0A25...,e1fa,598,79,96.056622,82.831785,96.794268,20db93f3257,True,452,True,2083812
7597992,/squash/af2_distillation_facebook/cif/c5/86/Un...,{['af2_distillation_facebook']}{UniRef50_A0A4Q...,08fd,1170,150,95.390611,74.535608,96.475673,c586cf83694,True,1167,True,1221644


Looks reasonable! Again, the most relevant columns for our `GenericDFParser` are the `path` and the `example_id` columns. In this case, we don't care about `assembly_id`, and all of our structures are below the token limit so we don't need to specify any query chains (`pn_unit_iid`)

Finally, let's wrap it up in a `StructuralDatasetWrapper` and we're done.

In [7]:
af2fb_metadata = PandasDataset(data=df_metadata, name="af2_distillation_facebook", id_column="example_id")

af2fb_dataset = StructuralDatasetWrapper(
    af2fb_metadata,
    dataset_parser=GenericDFParser(
        example_id_colname="example_id", path_colname="path", pn_unit_iid_colnames=[]
    ),  # <-- how to take the metadata and extract a path to the relevant files
    # ^ We must explicitly pass an empty list to `pn_unit_iid_colnames`, since the default behavior is to look for a column named `pn_unit_iid` in the metadata
    cif_parser_args={},  # <-- any additional arguments to pass to the CIF parser,
    save_failed_examples_to_dir=None,  # <-- whether to save failed examples to a directory, here we don't need that.
)

Let's see if we can load an example.

In [8]:
af2fb_dataset[0]

{'example_id': "{['af2_distillation_facebook']}{UniRef50_A0A0D2M3P0}{}{}",
 'path': PosixPath('/squash/af2_distillation_facebook/cif/60/84/UniRef50_A0A0D2M3P0.cif'),
 'assembly_id': '1',
 'query_pn_unit_iids': [],
 'extra_info': {'path': '/squash/af2_distillation_facebook/cif/60/84/UniRef50_A0A0D2M3P0.cif',
  'example_id': "{['af2_distillation_facebook']}{UniRef50_A0A0D2M3P0}{}{}",
  'original_hash_for_shard': '8bca',
  'n_atoms': 1402,
  'n_res': 187,
  'mean_plddt': 88.80805012258176,
  'min_plddt': 44.73389925580704,
  'median_plddt': 92.60271288105331,
  'sequence_hash': '608400c62c4',
  'has_msa': True,
  'msa_depth': 1517,
  'has_template': True,
  'cluster_id': 789965},
 'atom_array': AtomArray([
 	Atom(np.array([-14.933,   5.161, -14.817], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="MET", hetero=False, atom_name="N", element="N", charge=0, occupancy=1.0, alt_atom_id="N", uses_alt_atom_id=False, is_aromatic=False, b_factor=66.57382970859179, stereo="N", is_bac

We can also load by example ID.

In [9]:
af2fb_dataset.id_to_idx("{['af2_distillation_facebook']}{UniRef50_A0A0D2M3P0}{}{}")

0

In [10]:
af2fb_dataset[af2fb_dataset.id_to_idx("{['af2_distillation_facebook']}{UniRef50_A0A0D2M3P0}{}{}")]

{'example_id': "{['af2_distillation_facebook']}{UniRef50_A0A0D2M3P0}{}{}",
 'path': PosixPath('/squash/af2_distillation_facebook/cif/60/84/UniRef50_A0A0D2M3P0.cif'),
 'assembly_id': '1',
 'query_pn_unit_iids': [],
 'extra_info': {'path': '/squash/af2_distillation_facebook/cif/60/84/UniRef50_A0A0D2M3P0.cif',
  'example_id': "{['af2_distillation_facebook']}{UniRef50_A0A0D2M3P0}{}{}",
  'original_hash_for_shard': '8bca',
  'n_atoms': 1402,
  'n_res': 187,
  'mean_plddt': 88.80805012258176,
  'min_plddt': 44.73389925580704,
  'median_plddt': 92.60271288105331,
  'sequence_hash': '608400c62c4',
  'has_msa': True,
  'msa_depth': 1517,
  'has_template': True,
  'cluster_id': 789965},
 'atom_array': AtomArray([
 	Atom(np.array([-14.933,   5.161, -14.817], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="MET", hetero=False, atom_name="N", element="N", charge=0, occupancy=1.0, alt_atom_id="N", uses_alt_atom_id=False, is_aromatic=False, b_factor=66.57382970859179, stereo="N", is_bac

Finally, let's add a few transforms, just for fun

In [11]:
from datahub.transforms.base import Compose
from datahub.transforms.atom_array import AddGlobalAtomIdAnnotation
from datahub.transforms.atomize import AtomizeByCCDName
from datahub.encoding_definitions import RF2_ATOM36_ENCODING
from datahub.transforms.encoding import EncodeAtomArray
from datahub.transforms.crop import CropSpatialLikeAF3

af2fb_dataset = StructuralDatasetWrapper(
    af2fb_metadata,
    # WARNING: USE `GenericDFParser` INSTEAD OF CUSTOM PARSERS FOR EACH NEW DISTILLATION DATASET
    dataset_parser=GenericDFParser(
        example_id_colname="example_id", path_colname="path", pn_unit_iid_colnames=[]
    ),  # <-- how to take the metadata and extract a path to the relevant files
    cif_parser_args={},  # <-- any additional arguments to pass to the CIF parser,
    save_failed_examples_to_dir=None,  # <-- whether to save failed examples to a directory, here we don't need that.
    # These transforms will be applied to the data once loaded and the result will be returned
    transform=Compose(
        [
            AddGlobalAtomIdAnnotation(),
            AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=RF2_ATOM36_ENCODING.tokens),
            EncodeAtomArray(
                encoding=RF2_ATOM36_ENCODING,
            ),
            CropSpatialLikeAF3(crop_size=128),
        ]
    ),
)
af2fb_dataset[0]

Failed to create RF2 and RF2AA encodings. Falling back to the `frozen` defaults. If they have been updated in the meantime, you will need to get the above `try` to work. No module named 'rf2aa'


Invalid input for CropSpatialLikeAF3: Transform `CropSpatialLikeAF3` cannot be applied if any of the transforms ['EncodeAtomArray', 'CropContiguousLikeAF3', 'CropSpatialLikeAF3', 'PlaceUnresolvedTokenOnClosestResolvedTokenInSequence'] have been applied before it. Current transform history: ['load_example_from_metadata_row', 'AddGlobalAtomIdAnnotation', 'AtomizeByCCDName', 'EncodeAtomArray']

Traceback (most recent call last):
  File "/home/ncorley/projects/datahub/src/datahub/transforms/base.py", line 253, in __call__
    self._check_transform_history(data)
  File "/home/ncorley/projects/datahub/src/datahub/transforms/base.py", line 216, in _check_transform_history
    raise ValueError(
ValueError: Transform `CropSpatialLikeAF3` cannot be applied if any of the transforms ['EncodeAtomArray', 'CropContiguousLikeAF3', 'CropSpatialLikeAF3', 'PlaceUnresolvedTokenOnClosestResolvedTokenInSequence'] have been applied before it. Current transform history: ['load_example_from_metadata_row', 'Add

TransformPipelineError: Transforms failed at stage `CropSpatialLikeAF3`: Invalid input for CropSpatialLikeAF3: Transform `CropSpatialLikeAF3` cannot be applied if any of the transforms ['EncodeAtomArray', 'CropContiguousLikeAF3', 'CropSpatialLikeAF3', 'PlaceUnresolvedTokenOnClosestResolvedTokenInSequence'] have been applied before it. Current transform history: ['load_example_from_metadata_row', 'AddGlobalAtomIdAnnotation', 'AtomizeByCCDName', 'EncodeAtomArray']

Traceback (most recent call last):
  File "/home/ncorley/projects/datahub/src/datahub/transforms/base.py", line 253, in __call__
    self._check_transform_history(data)
  File "/home/ncorley/projects/datahub/src/datahub/transforms/base.py", line 216, in _check_transform_history
    raise ValueError(
ValueError: Transform `CropSpatialLikeAF3` cannot be applied if any of the transforms ['EncodeAtomArray', 'CropContiguousLikeAF3', 'CropSpatialLikeAF3', 'PlaceUnresolvedTokenOnClosestResolvedTokenInSequence'] have been applied befo

Oops! I did something stupid here -- I tried to first encode and then crop. That's dangerous because it could be that we 
crop mid-encoded-token or so (and it's also inefficient for large structures, because I only need to encode what's in the crop
since that's everything I'm gonna use anyways.). 
So the pipeline does not allow us to do that errors with a `TransformPipelineError`.

Let's fix it by first cropping and then encoding.

In [13]:
af2fb_dataset = StructuralDatasetWrapper(
    af2fb_metadata,
    # WARNING: USE `GenericDFParser` INSTEAD OF CUSTOM PARSERS FOR EACH NEW DISTILLATION DATASET
    dataset_parser=GenericDFParser(
        example_id_colname="example_id", path_colname="path", pn_unit_iid_colnames=[]
    ),  # <-- how to take the metadata and extract a path to the relevant files
    cif_parser_args={},  # <-- any additional arguments to pass to the CIF parser,
    save_failed_examples_to_dir=None,  # <-- whether to save failed examples to a directory, here we don't need that.
    # These transforms will be applied to the data once loaded and the result will be returned
    transform=Compose(
        [
            AddGlobalAtomIdAnnotation(),
            AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=RF2_ATOM36_ENCODING.tokens),
            CropSpatialLikeAF3(crop_size=128),
            EncodeAtomArray(
                encoding=RF2_ATOM36_ENCODING,
            ),
        ]
    ),
)
data = af2fb_dataset[0]

Let's take a peek at our crop:

In [14]:
from cifutils.utils.visualize import view

view(data["atom_array"])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Look reasonable!